# 04 - SQL Analytics

## Objective
Answer portfolio-level business questions in SQL, kept deliberately separate from
the Python feature code: the analytics layer describes the book, the Python layer
prepares model inputs.

Engine: **DuckDB** over the processed CSV, so the whole analysis reproduces with
no database server.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import duckdb
import pandas as pd

from src import config
from src.analytics import runner

## Configuration and data loading

In [ ]:
SQL_FILE = config.SQL_DIR / "analytics.sql"
loans = pd.read_csv(config.FEATURED_DATA_FILE)
connection = duckdb.connect(database=":memory:")
connection.register("loans", loans)
print(f"{len(loans):,} loans registered as the `loans` table")

## The query catalogue

Queries live in `sql/analytics.sql` and are split on their `-- name:` headers.

In [ ]:
queries = runner.parse_queries(SQL_FILE)
list(queries)

## Portfolio KPIs

In [ ]:
connection.execute(queries["01_portfolio_overview"]).fetchdf()

## Risk by customer characteristics

In [ ]:
for name in ["02_default_rate_by_employment", "03_default_rate_by_income_group",
             "06_default_rate_by_home_ownership", "07_default_rate_by_age_group"]:
    print(f"--- {name} ---")
    display(connection.execute(queries[name]).fetchdf())

## Risk by product and pricing

In [ ]:
for name in ["04_default_rate_by_loan_intent", "05_default_rate_by_credit_grade",
             "08_default_rate_by_prior_default", "09_loan_burden_bands"]:
    print(f"--- {name} ---")
    display(connection.execute(queries[name]).fetchdf())

## Portfolio concentration and watchlist

In [ ]:
for name in ["10_high_risk_customers", "11_portfolio_distribution_by_loan_size",
             "12_worst_segments_grade_x_intent"]:
    print(f"--- {name} ---")
    display(connection.execute(queries[name]).fetchdf())

## Ad-hoc query

The same connection supports free-form analysis.

In [ ]:
connection.execute("""
    SELECT loan_grade,
           COUNT(*)                         AS loans,
           ROUND(AVG(loan_int_rate), 2)     AS avg_rate,
           ROUND(AVG(loan_percent_income), 3) AS avg_burden,
           ROUND(AVG(loan_status) * 100, 2) AS default_rate_pct
    FROM loans
    WHERE person_income < 40000
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

## Results

The full set of results is also written to `reports/sql_analytics.md` by `run_pipeline.py`.

In [ ]:
results = runner.run_queries(loans, SQL_FILE)
report_path = runner.save_report(results)
print("Report written to", report_path)
connection.close()

## Interpretation
The SQL layer confirms, at portfolio level, the same three drivers the EDA found:
affordability (loan-to-income band), pricing/grade, and stability (employment,
bureau history). It also quantifies *exposure*, which the modelling metrics do not:
a segment can have a moderate default rate and still dominate expected loss because
its average ticket is large.

## Conclusion
These queries are what a risk or portfolio analyst would run monthly; the model in
notebook 05 turns the same signals into a customer-level probability.